# CutMind — train the Hinglish → English translator (free Colab GPU)

**What this does:** teaches a small open model (Qwen2.5-1.5B-Instruct, Apache-2.0) to turn casual Hindi / Hinglish
editing requests into the clean English instructions the CutMind editing brain understands. Then it exports the model
to ONNX int4 so it runs on a normal CPU server (no GPU bill when you go live).

**How to run (about 30–45 min, free):**
1. `Runtime → Change runtime type → T4 GPU` → Save.
2. `Runtime → Run all`.
3. When asked, upload `train.jsonl` and `eval.jsonl` from the project folder `training/data/`.
4. At the end a file `cutmind_translator.zip` downloads. Unzip it into
   `backend/assets/models/translator/` (so that `backend/assets/models/translator/onnx/model_q4.onnx` exists)
   and restart the backend. Then run `python -m cutmind.translator_bench trained` to see the score.

Nothing here needs a paid account; your data stays in your Colab session.

In [ ]:
# 1) libraries: training ones pinned (they work together); onnx / onnxruntime latest (ready-made for Colab's Python)
!pip -q install "transformers==4.46.3" "peft==0.13.2" "trl==0.12.1" "accelerate==1.1.1" "datasets==3.1.0" "optimum==1.23.3"
!pip -q install onnx onnxruntime
import torch, json, os, random, importlib
for lib in ("transformers", "peft", "trl", "datasets", "optimum", "onnx", "onnxruntime"):
    importlib.import_module(lib)  # stops here with a clear error if something did not install
print("libraries OK")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime → Change runtime type → T4 GPU")

In [ ]:
# 2) the data: train.jsonl and eval.jsonl from the project folder  training\data\
#    EITHER drag both files into the Files panel on the left (folder icon) before running,
#    OR click "Choose Files" below and select BOTH files (Ctrl + click) — not the notebook.
from google.colab import files
if not (os.path.exists("train.jsonl") and os.path.exists("eval.jsonl")):
    files.upload()
assert os.path.exists("train.jsonl") and os.path.exists("eval.jsonl"), "need train.jsonl AND eval.jsonl (see the note above)"
train = [json.loads(l) for l in open("train.jsonl", encoding="utf-8")]
evals = [json.loads(l) for l in open("eval.jsonl", encoding="utf-8")]
print(len(train), "train /", len(evals), "eval examples. e.g.", train[0])

In [ ]:
# 3) model + chat format (the same prompt the app sends)
from transformers import AutoTokenizer, AutoModelForCausalLM
BASE = "Qwen/Qwen2.5-1.5B-Instruct"   # Apache-2.0. Smaller/faster on CPU: "Qwen/Qwen2.5-0.5B-Instruct"
SYSTEM = ("You translate requests for a video editing app from Hindi / Hinglish (Hindi written in English letters) "
          "into clear, simple English editing instructions. Keep numbers, seconds, colours, names and quoted text "
          "exactly. Translate what is asked, nothing more. Answer with the English only, on one line.")
tok = AutoTokenizer.from_pretrained(BASE)
def to_chat(ex):
    return {"messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["input"]},
                         {"role": "assistant", "content": ex["output"]}]}
from datasets import Dataset
ds_train = Dataset.from_list([to_chat(x) for x in train])
ds_eval = Dataset.from_list([to_chat(x) for x in evals[:300]])
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")

In [ ]:
# 4) LoRA fine-tuning (~20-30 min on a T4)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
# small batches + gradient checkpointing: fits the free T4 (15 GB); 4 x 4 = the same 16 examples per step as before
cfg = SFTConfig(output_dir="out", num_train_epochs=2, per_device_train_batch_size=4, gradient_accumulation_steps=4,
                per_device_eval_batch_size=4, gradient_checkpointing=True, learning_rate=2e-4, lr_scheduler_type="cosine",
                warmup_ratio=0.03, logging_steps=25, eval_strategy="steps", eval_steps=200, save_strategy="no", fp16=True,
                max_seq_length=320, report_to="none")
model.config.use_cache = False  # needed with gradient checkpointing
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds_train, eval_dataset=ds_eval, peft_config=lora,
                     tokenizer=tok)
trainer.train()

In [ ]:
# 5) quick check on held-out examples (exact match with the expected English)
model = trainer.model.merge_and_unload()
model.eval()
def translate(text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=48, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip().split("\n")[0]
sample = random.Random(1).sample(evals, min(100, len(evals)))
hits = sum(translate(x["input"]) == x["output"] for x in sample)
print(f"exact match on {len(sample)} held-out examples: {hits}")
for t in ["rangon ko thoda halka sa kar do", "video ko piche ki taraf chalao", "neeli t-shirt wale ladke pe focus, baaki sab blur",
          "kone kale kar do thode", "video bahut andhera hai, thoda ujla karo", "mera kutta cute hai"]:
    print(t, "->", translate(t))

In [ ]:
# 6) save → ONNX (with KV cache) → int4 for CPU, then zip and download
import os, glob
if not os.path.exists("merged/config.json"):          # the trained model (saved by the training cell)
    model.save_pretrained("merged", safe_serialization=True)
    tok.save_pretrained("merged")
!pip -q uninstall -y diffusers                          # Colab's diffusers breaks optimum's exporter
!pip -q install onnxscript                              # newer PyTorch asks for it
import torch
import optimum.exporters.onnx.convert as conv
_orig_export = torch.onnx.export
def _classic_export(*a, **k):                           # the classic exporter optimum was built for
    k["dynamo"] = False
    return _orig_export(*a, **k)
torch.onnx.export = _classic_export
conv.onnx_export = _classic_export
from optimum.exporters.onnx import main_export
!rm -rf export_fp32
main_export("merged", output="export_fp32", task="text-generation-with-past", framework="pt", device="cpu")
assert os.path.exists("export_fp32/model.onnx"), "ONNX export failed - send me the red text above"

import onnx
try:                                                    # name in newer onnxruntime
    from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer as Q
    make_q = lambda m: Q(m, bits=4, block_size=32, is_symmetric=True)
except ImportError:                                     # name in older onnxruntime
    from onnxruntime.quantization.matmul_4bits_quantizer import MatMul4BitsQuantizer as Q
    make_q = lambda m: Q(m, block_size=32, is_symmetric=True)

os.makedirs("translator/onnx", exist_ok=True)
m = onnx.load("export_fp32/model.onnx", load_external_data=True)
q = make_q(m)
q.process()
q.model.save_model_to_file("translator/onnx/model_q4.onnx", use_external_data_format=True)
for f in ("tokenizer.json", "config.json", "tokenizer_config.json", "generation_config.json"):
    if os.path.exists(f"merged/{f}"):
        os.system(f"cp merged/{f} translator/")
print(sorted(glob.glob("translator/**", recursive=True)))
!cd translator && zip -r -q ../cutmind_translator.zip . && cd .. && ls -la cutmind_translator.zip
from google.colab import files
files.download("cutmind_translator.zip")